# Data and provenance

What was downloaded, from where, and what the data profile found.

This notebook only presents results. Every number below was computed by the pipeline (`make all`) and is read from `results/` or from the warehouse through a read-only connection; nothing here trains a model or recomputes a reported number. Run it with `make notebooks`.

In [ ]:
profile = "full"

In [ ]:
import duckdb
import pandas as pd
from IPython.display import Image, display

from cigd.config import REPOSITORY_ROOT, load_config

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
config = load_config(profile)


def table(name: str) -> pd.DataFrame:
    """Read one result table written by the pipeline."""
    return pd.read_csv(config.tables_dir / f"{name}.csv")


def figure(name: str) -> None:
    """Show one figure written by the pipeline."""
    display(Image(filename=str(config.figures_dir / f"{name}.png"), width=900))


def warehouse_query(sql: str) -> pd.DataFrame:
    """Query the warehouse through a read-only connection."""
    with duckdb.connect(str(config.warehouse_path), read_only=True) as connection:
        return connection.execute(sql).df()

## Downloads by source

From `provenance/manifest.csv`. Historical trip months are counted and deleted, so only their aggregate is stored.

In [ ]:
table("integration_raw_storage")

## Sources at a glance

In [ ]:
table("profile_sources")

## Yellow taxi trips by month, as published

Quality counts before any cleaning. The validity rules that act on these are in notebook 02.

In [ ]:
table("profile_trip_months")

In [ ]:
table("profile_trip_schema_changes")

## NYSERDA weekly price panel

Every missing week is documented in `reference/known_price_gaps.csv`; flagged prices are kept, not removed.

In [ ]:
table("profile_price_series")

In [ ]:
table("profile_price_missing_weeks")

In [ ]:
table("profile_price_outliers")

## EIA cross-check

The original level check (fixed before profiling) and the weekly change check added on 2026-10-09 (see `docs/decisions.md`).

In [ ]:
table("profile_eia_crosscheck")

In [ ]:
table("profile_eia_change_check")

## Weather

In [ ]:
table("profile_weather")

## Publication-lag evidence

When each TLC month and each NYSERDA label was first published, from Internet Archive snapshots (`docs/evidence/`).

In [ ]:
pd.read_csv(REPOSITORY_ROOT / "docs" / "evidence" / "tlc_release_dates.csv")

In [ ]:
pd.read_csv(REPOSITORY_ROOT / "docs" / "evidence" / "nyserda_release_dates.csv")

## The two domains over time

In [ ]:
figure("data_price_and_demand")